In [1]:
# ============================================
# 🤖 MACHINE LEARNING — Prédiction Prix Immobilier
# ============================================
# Modèle: XGBoost Regressor
# Objectif: Prédire le prix d'un bien immobilier
# Auteur: Walid Bourhaleb
# ============================================

# Upload du fichier
from google.colab import files
print("📁 Sélectionnez 'sales_clean_with_features.csv'")
uploaded = files.upload()

📁 Sélectionnez 'sales_clean_with_features.csv'


Saving sales_clean_with_features.csv to sales_clean_with_features.csv


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

# Charger les données
df = pd.read_csv('sales_clean_with_features.csv', low_memory=False)
df['date_mutation'] = pd.to_datetime(df['date_mutation'], errors='coerce')

print(f"📊 Dataset: {len(df):,} ventes, {len(df.columns)} colonnes")
print(f"📅 Période: {df['date_mutation'].min()} → {df['date_mutation'].max()}")

📊 Dataset: 16,184 ventes, 49 colonnes
📅 Période: 2023-01-02 00:00:00 → 2023-12-06 00:00:00


In [3]:
# ============================================
# 🔧 PRÉPARATION DES FEATURES
# ============================================

# Garder seulement les biens avec données complètes
df_ml = df.dropna(subset=['valeur_fonciere', 'surface_reelle_bati', 'type_local']).copy()
df_ml = df_ml[df_ml['type_local'].isin(['Appartement', 'Maison'])]
df_ml = df_ml[df_ml['valeur_fonciere'] > 0]
df_ml = df_ml[df_ml['surface_reelle_bati'] > 0]

print(f"📊 Données pour ML: {len(df_ml):,} ventes")

# Définir les features
feature_columns = [
    'surface_reelle_bati',
    'nb_pieces',
    'type_local_encoded',
    'mois',
    'trimestre',
    'jour_semaine',
]

# Ajouter les features optionnelles (si elles existent et ne sont pas vides)
optional_features = [
    'distance_centre_ville',
    'commune_prix_m2_moyen',
    'commune_nb_ventes',
    'commune_population',
    'commune_densite',
    'grande_surface',
    'proche_centre',
]

for feat in optional_features:
    if feat in df_ml.columns:
        non_null_pct = df_ml[feat].notna().mean()
        if non_null_pct > 0.3:  # Au moins 30% de données
            feature_columns.append(feat)
            print(f"  ✅ {feat} ajouté ({non_null_pct*100:.0f}% complet)")
        else:
            print(f"  ⚠️ {feat} ignoré ({non_null_pct*100:.0f}% complet)")

# Convertir les booléens en int
for col in df_ml.columns:
    if df_ml[col].dtype == 'bool':
        df_ml[col] = df_ml[col].astype(int)

# Target (ce qu'on prédit)
target = 'valeur_fonciere'

# Supprimer les lignes avec des NaN dans les features
df_ml = df_ml.dropna(subset=feature_columns)

print(f"\n📋 Features sélectionnées ({len(feature_columns)}):")
for f in feature_columns:
    print(f"  • {f}")
print(f"\n🎯 Target: {target}")
print(f"📊 Données finales: {len(df_ml):,} ventes")

📊 Données pour ML: 16,183 ventes
  ✅ distance_centre_ville ajouté (65% complet)
  ✅ commune_prix_m2_moyen ajouté (65% complet)
  ✅ commune_nb_ventes ajouté (65% complet)
  ✅ commune_population ajouté (65% complet)
  ✅ commune_densite ajouté (65% complet)
  ✅ grande_surface ajouté (100% complet)
  ✅ proche_centre ajouté (65% complet)

📋 Features sélectionnées (13):
  • surface_reelle_bati
  • nb_pieces
  • type_local_encoded
  • mois
  • trimestre
  • jour_semaine
  • distance_centre_ville
  • commune_prix_m2_moyen
  • commune_nb_ventes
  • commune_population
  • commune_densite
  • grande_surface
  • proche_centre

🎯 Target: valeur_fonciere
📊 Données finales: 10,470 ventes


In [4]:
# ============================================
# ✂️ SPLIT TRAIN / TEST
# ============================================

X = df_ml[feature_columns].values
y = df_ml[target].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"📊 Split des données:")
print(f"  Train: {len(X_train):,} ({len(X_train)/len(X)*100:.0f}%)")
print(f"  Test:  {len(X_test):,} ({len(X_test)/len(X)*100:.0f}%)")
print(f"\n💰 Prix dans le train set:")
print(f"  Moyen: {y_train.mean():,.0f} €")
print(f"  Médian: {np.median(y_train):,.0f} €")
print(f"  Min: {y_train.min():,.0f} €")
print(f"  Max: {y_train.max():,.0f} €")

📊 Split des données:
  Train: 8,376 (80%)
  Test:  2,094 (20%)

💰 Prix dans le train set:
  Moyen: 336,663 €
  Médian: 238,000 €
  Min: 11,000 €
  Max: 9,800,000 €


In [5]:
# ============================================
# 🏋️ ENTRAÎNEMENT DE PLUSIEURS MODÈLES
# ============================================

models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(
        n_estimators=100, max_depth=15, random_state=42, n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=100, max_depth=10, learning_rate=0.1, random_state=42
    ),
    "XGBoost": xgb.XGBRegressor(
        n_estimators=200, max_depth=15, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        min_child_weight=5, gamma=0.1,
        random_state=42, n_jobs=-1
    ),
}

results = {}

print("🏋️ Entraînement des modèles...\n")
print(f"{'Modèle':<25} {'R²':>8} {'MAE':>12} {'RMSE':>12} {'Temps':>8}")
print("-" * 70)

import time

for name, model in models.items():
    start = time.time()

    # Entraîner
    model.fit(X_train, y_train)

    # Prédire
    y_pred = model.predict(X_test)

    # Métriques
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    elapsed = time.time() - start

    results[name] = {
        'model': model,
        'r2': r2,
        'mae': mae,
        'rmse': rmse,
        'y_pred': y_pred,
        'time': elapsed
    }

    print(f"{name:<25} {r2:>8.4f} {mae:>11,.0f}€ {rmse:>11,.0f}€ {elapsed:>7.1f}s")

# Meilleur modèle
best_name = max(results, key=lambda x: results[x]['r2'])
print(f"\n🏆 Meilleur modèle: {best_name} (R² = {results[best_name]['r2']:.4f})")

🏋️ Entraînement des modèles...

Modèle                          R²          MAE         RMSE    Temps
----------------------------------------------------------------------
Linear Regression           0.1110     150,848€     503,209€     0.0s
Random Forest               0.6381     122,178€     321,065€     4.5s
Gradient Boosting           0.6231     120,690€     327,671€     3.7s
XGBoost                     0.5888     131,429€     342,233€     1.4s

🏆 Meilleur modèle: Random Forest (R² = 0.6381)


In [6]:
# ============================================
# 📊 COMPARAISON DES MODÈLES
# ============================================

model_names = list(results.keys())
r2_scores = [results[m]['r2'] for m in model_names]
mae_scores = [results[m]['mae'] for m in model_names]

fig = make_subplots(rows=1, cols=2,
    subplot_titles=("R² Score (plus haut = mieux)", "MAE en € (plus bas = mieux)"))

colors = ['#94a3b8', '#60a5fa', '#f59e0b', '#22c55e']

fig.add_trace(
    go.Bar(x=model_names, y=r2_scores, marker_color=colors, name='R²',
           text=[f"{r:.3f}" for r in r2_scores], textposition='outside'),
    row=1, col=1
)

fig.add_trace(
    go.Bar(x=model_names, y=mae_scores, marker_color=colors, name='MAE',
           text=[f"{m:,.0f}€" for m in mae_scores], textposition='outside'),
    row=1, col=2
)

fig.update_layout(
    title="📊 Comparaison des modèles de prédiction",
    height=450, template='plotly_white', showlegend=False
)
fig.show()

In [7]:
# ============================================
# 🏆 ANALYSE DU MEILLEUR MODÈLE
# ============================================

best = results[best_name]
y_pred_best = best['y_pred']

# Scatter plot: Réel vs Prédit
fig = px.scatter(
    x=y_test, y=y_pred_best,
    opacity=0.3,
    title=f'🎯 {best_name}: Prix réel vs Prix prédit',
    labels={'x': 'Prix réel (€)', 'y': 'Prix prédit (€)'}
)

# Ligne parfaite (y=x)
max_val = max(y_test.max(), y_pred_best.max())
fig.add_trace(go.Scatter(
    x=[0, max_val], y=[0, max_val],
    mode='lines', name='Prédiction parfaite',
    line=dict(color='red', dash='dash')
))

fig.update_layout(height=500, template='plotly_white')
fig.show()

# Distribution des erreurs
errors = y_test - y_pred_best
error_pct = (errors / y_test) * 100

fig = go.Figure()
fig.add_trace(go.Histogram(x=error_pct, nbinsx=50, marker_color='#3b82f6'))
fig.update_layout(
    title=f'📊 Distribution des erreurs de prédiction ({best_name})',
    xaxis_title='Erreur (%)',
    yaxis_title='Nombre de ventes',
    height=400, template='plotly_white'
)
fig.show()

print(f"\n📊 Statistiques des erreurs:")
print(f"  Erreur moyenne: {np.mean(np.abs(error_pct)):.1f}%")
print(f"  Erreur médiane: {np.median(np.abs(error_pct)):.1f}%")
print(f"  90% des prédictions à ±{np.percentile(np.abs(error_pct), 90):.0f}%")


📊 Statistiques des erreurs:
  Erreur moyenne: 49.0%
  Erreur médiane: 24.3%
  90% des prédictions à ±95%


In [8]:
# ============================================
# 📊 FEATURE IMPORTANCE
# ============================================

best_model = best['model']

if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
    feat_imp = pd.DataFrame({
        'feature': feature_columns,
        'importance': importances
    }).sort_values('importance', ascending=True)

    fig = px.bar(feat_imp, x='importance', y='feature',
                 orientation='h',
                 color='importance',
                 color_continuous_scale='Blues',
                 title=f'📊 Importance des features ({best_name})')

    fig.update_layout(height=400, template='plotly_white')
    fig.show()

    print("\n📊 Top features:")
    for _, row in feat_imp.sort_values('importance', ascending=False).head(10).iterrows():
        bar = "█" * int(row['importance'] * 50)
        print(f"  {row['feature']:<30} {row['importance']:.4f} {bar}")


📊 Top features:
  surface_reelle_bati            0.2219 ███████████
  jour_semaine                   0.2130 ██████████
  commune_prix_m2_moyen          0.1787 ████████
  mois                           0.1386 ██████
  commune_densite                0.0839 ████
  trimestre                      0.0613 ███
  nb_pieces                      0.0398 █
  commune_population             0.0285 █
  commune_nb_ventes              0.0272 █
  type_local_encoded             0.0066 


In [9]:
# ============================================
# 🔄 CROSS-VALIDATION (5-fold)
# ============================================

print(f"🔄 Cross-validation 5-fold pour {best_name}...")

cv_scores = cross_val_score(
    best_model, X, y,
    cv=5, scoring='r2', n_jobs=-1
)

print(f"\n📊 Résultats Cross-Validation:")
print(f"  R² scores: {[f'{s:.4f}' for s in cv_scores]}")
print(f"  R² moyen: {cv_scores.mean():.4f}")
print(f"  R² écart-type: {cv_scores.std():.4f}")
print(f"  R² intervalle: [{cv_scores.mean()-2*cv_scores.std():.4f}, {cv_scores.mean()+2*cv_scores.std():.4f}]")

fig = go.Figure()
fig.add_trace(go.Bar(
    x=[f'Fold {i+1}' for i in range(5)],
    y=cv_scores,
    marker_color=['#3b82f6', '#22c55e', '#f59e0b', '#ef4444', '#a855f7'],
    text=[f'{s:.4f}' for s in cv_scores],
    textposition='outside'
))
fig.add_hline(y=cv_scores.mean(), line_dash="dash", line_color="red",
              annotation_text=f"Moyenne: {cv_scores.mean():.4f}")
fig.update_layout(
    title='🔄 Cross-Validation R² Scores',
    yaxis_title='R² Score',
    height=400, template='plotly_white'
)
fig.show()

🔄 Cross-validation 5-fold pour Random Forest...

📊 Résultats Cross-Validation:
  R² scores: ['0.3421', '0.3619', '0.2879', '-0.0448', '0.1651']
  R² moyen: 0.2224
  R² écart-type: 0.1502
  R² intervalle: [-0.0780, 0.5228]


In [10]:
# ============================================
# 💾 SAUVEGARDER LE MODÈLE
# ============================================

import pickle

# Sauvegarder le modèle + metadata
model_data = {
    'model': best_model,
    'model_name': best_name,
    'feature_columns': feature_columns,
    'metrics': {
        'r2': best['r2'],
        'mae': best['mae'],
        'rmse': best['rmse'],
        'cv_r2_mean': cv_scores.mean(),
        'cv_r2_std': cv_scores.std(),
    },
    'training_samples': len(X_train),
    'test_samples': len(X_test),
}

# Sauvegarder en pickle
with open('price_predictor_v1.pkl', 'wb') as f:
    pickle.dump(model_data, f)

print("💾 Modèle sauvegardé: price_predictor_v1.pkl")
print(f"\n📊 Résumé du modèle:")
print(f"  Nom: {best_name}")
print(f"  R²: {best['r2']:.4f}")
print(f"  MAE: {best['mae']:,.0f} €")
print(f"  RMSE: {best['rmse']:,.0f} €")
print(f"  CV R² moyen: {cv_scores.mean():.4f}")
print(f"  Features: {len(feature_columns)}")
print(f"  Train: {len(X_train):,} / Test: {len(X_test):,}")

# Télécharger le modèle
files.download('price_predictor_v1.pkl')
print("\n📥 Téléchargement du modèle lancé !")

💾 Modèle sauvegardé: price_predictor_v1.pkl

📊 Résumé du modèle:
  Nom: Random Forest
  R²: 0.6381
  MAE: 122,178 €
  RMSE: 321,065 €
  CV R² moyen: 0.2224
  Features: 13
  Train: 8,376 / Test: 2,094


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


📥 Téléchargement du modèle lancé !


In [11]:
# ============================================
# 🎯 DÉMO: PRÉDIRE LE PRIX D'UN BIEN
# ============================================

def predict_price(surface, nb_pieces, type_bien='Appartement', **kwargs):
    """
    Prédit le prix d'un bien immobilier.

    Args:
        surface: Surface en m²
        nb_pieces: Nombre de pièces
        type_bien: 'Appartement' ou 'Maison'
        **kwargs: Autres features optionnelles
    """
    # Préparer les features
    features = {}

    # Features obligatoires
    features['surface_reelle_bati'] = surface
    features['nb_pieces'] = nb_pieces
    features['type_local_encoded'] = 1 if type_bien == 'Appartement' else 2
    features['mois'] = kwargs.get('mois', 6)
    features['trimestre'] = kwargs.get('trimestre', 2)
    features['jour_semaine'] = kwargs.get('jour_semaine', 3)

    # Features optionnelles
    for feat in feature_columns:
        if feat not in features:
            if feat in kwargs:
                features[feat] = kwargs[feat]
            else:
                # Utiliser la médiane du dataset
                if feat in df_ml.columns:
                    features[feat] = df_ml[feat].median()
                else:
                    features[feat] = 0

    # Créer le vecteur de features dans le bon ordre
    X_pred = np.array([[features[f] for f in feature_columns]])

    # Prédire
    prix = best_model.predict(X_pred)[0]

    # Intervalle de confiance approximatif (±RMSE)
    rmse = best['rmse']
    prix_min = prix - 1.96 * rmse * 0.5
    prix_max = prix + 1.96 * rmse * 0.5

    return {
        'prix_predit': round(prix, 0),
        'prix_min': round(max(0, prix_min), 0),
        'prix_max': round(prix_max, 0),
        'prix_m2_estime': round(prix / surface, 0),
    }

# Exemples de prédictions
print("🎯 EXEMPLES DE PRÉDICTIONS\n")

exemples = [
    {"surface": 30, "nb_pieces": 1, "type": "Appartement", "desc": "Studio 30m² Paris"},
    {"surface": 55, "nb_pieces": 2, "type": "Appartement", "desc": "T2 55m² Lyon"},
    {"surface": 80, "nb_pieces": 3, "type": "Appartement", "desc": "T3 80m² Marseille"},
    {"surface": 120, "nb_pieces": 5, "type": "Maison", "desc": "Maison 120m² Bordeaux"},
    {"surface": 200, "nb_pieces": 7, "type": "Maison", "desc": "Grande maison 200m²"},
]

for ex in exemples:
    result = predict_price(ex['surface'], ex['nb_pieces'], ex['type'])
    print(f"🏠 {ex['desc']}:")
    print(f"   💶 Prix prédit: {result['prix_predit']:,.0f} €")
    print(f"   📊 Intervalle: {result['prix_min']:,.0f}€ — {result['prix_max']:,.0f}€")
    print(f"   📐 Prix/m² estimé: {result['prix_m2_estime']:,.0f} €/m²")
    print()

🎯 EXEMPLES DE PRÉDICTIONS

🏠 Studio 30m² Paris:
   💶 Prix prédit: 127,213 €
   📊 Intervalle: 0€ — 441,857€
   📐 Prix/m² estimé: 4,240 €/m²

🏠 T2 55m² Lyon:
   💶 Prix prédit: 186,660 €
   📊 Intervalle: 0€ — 501,303€
   📐 Prix/m² estimé: 3,394 €/m²

🏠 T3 80m² Marseille:
   💶 Prix prédit: 280,881 €
   📊 Intervalle: 0€ — 595,524€
   📐 Prix/m² estimé: 3,511 €/m²

🏠 Maison 120m² Bordeaux:
   💶 Prix prédit: 482,055 €
   📊 Intervalle: 167,412€ — 796,698€
   📐 Prix/m² estimé: 4,017 €/m²

🏠 Grande maison 200m²:
   💶 Prix prédit: 836,146 €
   📊 Intervalle: 521,502€ — 1,150,789€
   📐 Prix/m² estimé: 4,181 €/m²



In [12]:
# ============================================
# 📝 RÉSUMÉ FINAL
# ============================================

print("=" * 60)
print("🤖 RÉSUMÉ — MODÈLE DE PRÉDICTION DE PRIX")
print("=" * 60)
print(f"""
📊 DONNÉES:
  • {len(df_ml):,} ventes utilisées
  • {len(feature_columns)} features
  • Train: {len(X_train):,} / Test: {len(X_test):,}

🏆 MEILLEUR MODÈLE: {best_name}
  • R² Score: {best['r2']:.4f}
  • MAE: {best['mae']:,.0f} € (erreur moyenne)
  • RMSE: {best['rmse']:,.0f} €
  • CV R² moyen: {cv_scores.mean():.4f} (±{cv_scores.std():.4f})

📊 TOUS LES MODÈLES:
""")

for name, res in sorted(results.items(), key=lambda x: x[1]['r2'], reverse=True):
    star = " 🏆" if name == best_name else ""
    print(f"  {name:<25} R²={res['r2']:.4f}  MAE={res['mae']:,.0f}€{star}")

print(f"""
📋 TOP FEATURES:
""")
if hasattr(best_model, 'feature_importances_'):
    for feat, imp in sorted(zip(feature_columns, best_model.feature_importances_),
                            key=lambda x: x[1], reverse=True)[:5]:
        print(f"  • {feat}: {imp:.4f}")

print(f"""
💾 FICHIERS:
  • Modèle: price_predictor_v1.pkl
  • À placer dans: models/price_predictor_v1.pkl

✅ Prêt pour le dashboard Streamlit !
""")
print("=" * 60)

🤖 RÉSUMÉ — MODÈLE DE PRÉDICTION DE PRIX

📊 DONNÉES:
  • 10,470 ventes utilisées
  • 13 features
  • Train: 8,376 / Test: 2,094

🏆 MEILLEUR MODÈLE: Random Forest
  • R² Score: 0.6381
  • MAE: 122,178 € (erreur moyenne)
  • RMSE: 321,065 €
  • CV R² moyen: 0.2224 (±0.1502)

📊 TOUS LES MODÈLES:

  Random Forest             R²=0.6381  MAE=122,178€ 🏆
  Gradient Boosting         R²=0.6231  MAE=120,690€
  XGBoost                   R²=0.5888  MAE=131,429€
  Linear Regression         R²=0.1110  MAE=150,848€

📋 TOP FEATURES:

  • surface_reelle_bati: 0.2219
  • jour_semaine: 0.2130
  • commune_prix_m2_moyen: 0.1787
  • mois: 0.1386
  • commune_densite: 0.0839

💾 FICHIERS:
  • Modèle: price_predictor_v1.pkl
  • À placer dans: models/price_predictor_v1.pkl

✅ Prêt pour le dashboard Streamlit !

